In [ ]:
# Upload datasets from your system
from google.colab import files
uploaded = files.upload()

Saving sentiment_results.csv to sentiment_results.csv
Saving ratings_cleaned.csv to ratings_cleaned.csv


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np

In [ ]:
# Load cleaned rating data
ratings = pd.read_csv("ratings_cleaned.csv")

# Load sentiment model output
sentiments = pd.read_csv("sentiment_results.csv")

# Display sample data
print("Ratings Data Sample:")
display(ratings.head())

print("\nSentiments Data Sample:")
display(sentiments.head())

Ratings Data Sample:


,userId,movieId,rating,date,Month,Month Name,Year
0,1,1,4.0,19-06-2000 18:08,6.0,June,2000.0
1,1,3,4.0,19-06-2000 18:27,6.0,June,2000.0
2,1,17,4.0,20-06-2000 15:10,6.0,June,2000.0
3,1,21,4.0,19-06-2000 18:09,6.0,June,2000.0
4,1,39,4.0,19-06-2000 18:11,6.0,June,2000.0



Sentiments Data Sample:


,userId,movieId,tag,date,sentiment,predicted_sentiment
0,2,89774,tom hardy,24-10-2015 19:33,Neutral,Neutral
1,2,89774,boxing story,24-10-2015 19:33,Neutral,Neutral
2,2,106782,martin scorsese,24-10-2015 19:30,Neutral,Neutral
3,2,106782,leonardo dicaprio,24-10-2015 19:30,Neutral,Neutral
4,2,106782,drugs,24-10-2015 19:30,Neutral,Neutral


In [ ]:
# Convert text to lowercase and remove extra spaces
sentiments["predicted_sentiment"] = sentiments["predicted_sentiment"].astype(str).str.strip().str.lower()

# Fix common spelling mistake (nuetral → neutral)
sentiments["predicted_sentiment"] = sentiments["predicted_sentiment"].replace({
    "nuetral": "neutral"
})

# Check unique values to confirm cleaning
print("Unique Sentiment Values:")
print(sentiments["predicted_sentiment"].unique())

Unique Sentiment Values:
['neutral' 'positive' 'negative']


In [ ]:
# Map sentiment labels to numeric values
# positive = 1, neutral = 0, negative = -1
sentiment_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

# Apply mapping
sentiments["sentiment_score"] = sentiments["predicted_sentiment"].map(sentiment_map)

# Check for missing values after mapping
print("Missing sentiment_score values:", sentiments["sentiment_score"].isna().sum())

Missing sentiment_score values: 0


In [ ]:
# Calculate average rating and rating variance for each movie
# Average rating per movie
avg_rating = ratings.groupby("movieId")["rating"].mean().reset_index()
avg_rating.rename(columns={"rating": "avg_rating"}, inplace=True)

# Rating variance (disagreement)
rating_variance = ratings.groupby("movieId")["rating"].var().reset_index()
rating_variance.rename(columns={"rating": "rating_variance"}, inplace=True)

print("Avearge Ratings:")
display(avg_rating.head())

Avearge Ratings:


,movieId,avg_rating
0,1,3.937500
1,2,3.458716
2,3,3.313725
3,4,2.357143
4,5,3.125000


In [ ]:
# Calculate average sentiment score for each movie
avg_sentiment = sentiments.groupby("movieId")["sentiment_score"].mean().reset_index()
avg_sentiment.rename(columns={"sentiment_score": "avg_sentiment"}, inplace=True)

print("average Sentiments:")
display(avg_sentiment.head())

average Sentiments:


,movieId,avg_sentiment
0,1,0.333333
1,2,0.250000
2,3,0.000000
3,5,0.000000
4,7,0.000000


In [ ]:
# Merge both tables using movieId
# LEFT JOIN ensures all movies from ratings are kept
movie_insights = pd.merge(avg_rating, avg_sentiment, on="movieId", how="left")

movie_insights = pd.merge(movie_insights, rating_variance, on="movieId", how="left")
print("Merged Data:")
display(movie_insights.head())

Merged Data:


,movieId,avg_rating,avg_sentiment,rating_variance
0,1,3.937500,0.333333,0.648401
1,2,3.458716,0.250000,0.704298
2,3,3.313725,0.000000,0.979608
3,4,2.357143,NaN,0.726190
4,5,3.125000,0.000000,0.696809


In [ ]:
# Replace missing sentiment values with neutral (0)
movie_insights["avg_sentiment"] = movie_insights["avg_sentiment"].fillna(
    movie_insights["avg_sentiment"].mean()
)
# Replace missing variance (if only one rating exists)
movie_insights["rating_variance"] = movie_insights["rating_variance"].fillna(0)

In [ ]:
# Normalize rating from 0–5 scale to 0–1
movie_insights["normalized_rating"] = movie_insights["avg_rating"] / 5

# Normalize sentiment from -1 to 1 scale into 0–1
movie_insights["normalized_sentiment"] = (movie_insights["avg_sentiment"] + 1) / 2

In [ ]:
# Fairness Score = difference between rating and sentiment
movie_insights["fairness_score"] = abs(
    movie_insights["normalized_rating"] - movie_insights["normalized_sentiment"]
)

print("Fairness Score Calculated:")
display(movie_insights.head())

Fairness Score Calculated:


,movieId,avg_rating,avg_sentiment,rating_variance,normalized_rating,normalized_sentiment,fairness_score
0,1,3.937500,0.333333,0.648401,0.787500,0.666667,0.120833
1,2,3.458716,0.250000,0.704298,0.691743,0.625000,0.066743
2,3,3.313725,0.000000,0.979608,0.662745,0.500000,0.162745
3,4,2.357143,0.012885,0.726190,0.471429,0.506443,0.035014
4,5,3.125000,0.000000,0.696809,0.625000,0.500000,0.125000


In [ ]:
# Classification function
def classify_movie(row):

    # Controversial (high disagreement)
    if row["rating_variance"] > 1.2:
        return "Controversial"

    # Difference between sentiment & rating
    diff = row["normalized_sentiment"] - row["normalized_rating"]

    # Underrated
    if diff > 0.20:
        return "Underrated"

    # fair
    if -0.20 <= diff <= 0.20:
        return "Fair"

    # overrated
    return "Overrated"

# Apply classification
movie_insights["category"] = movie_insights.apply(classify_movie, axis=1)

In [ ]:
# Select only required columns for final output
movie_insights_final = movie_insights[[
    "movieId",
    "avg_rating",
    "avg_sentiment",
    "rating_variance",
    "fairness_score",
    "category"
]]

# Round values for clean output
movie_insights_final = movie_insights_final.round(3)

print("Final Movie Insights Table:")
display(movie_insights_final.head())

In [ ]:
# Save final result to CSV
movie_insights_final.to_csv("movie_insights.csv", index=False)

print("movie_insights.csv created successfully!")

movie_insights.csv created successfully!


In [ ]:
# Download the generated file
files.download("movie_insights.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
movie_insights["category"].value_counts()

,count
category,
Fair,4739
Overrated,3651
Controversial,833
Underrated,408
